# KMeans Optimization for Nods Classification in Spectroscopic Data Reduction

In [91]:
import numpy as np
from astropy.io import fits
import os
import shutil
import matplotlib.pyplot as plt
import glob
from scipy.signal import find_peaks
from sklearn.cluster import KMeans
import sklearn.preprocessing
from sklearn.metrics import accuracy_score, confusion_matrix, silhouette_score, precision_score, recall_score

### Building functions to run with multiple datasets

In [41]:
def classify_AB_frames(frames):
    collapsed = []
    center = []
    width = []
    total_flux = []
    peak = []
    asymmetry = []
    
    # feature engineering!
    for j, frame in enumerate(frames):
        spec = np.nanmedian(frame, axis=0) # collapsed frame
        collapsed.append(spec)
    
        spatial_profile = np.nanmedian(frame, axis=1) # collapsed spatial profile
        x = np.arange(len(spatial_profile))
        denom = np.nansum(spatial_profile)
        
        if not np.isfinite(denom) or (denom == 0):  # avoiding errors with undefined denominators when computing centroid
            print(f"Bad spatial profile in frame {j}, denom = {denom}")
            center.append(np.nan)
            width.append(np.nan)
            total_flux.append(np.nan)
            peak.append(np.nan)
            asymmetry.append(np.nan)
    
        else:
            cen = np.nansum(x * spatial_profile) / denom   # calculating the flux-weighted centroid
            
            wid = np.sqrt(np.nansum(spatial_profile * (x - cen)**2) / denom)  # calculating the general width of each frame
            
            pk = np.nanargmax(spatial_profile)  # finding the overall greatest peak of each frame
            
            # asymmetry calculations---toward left or right of avg. per frame
            mid = len(spatial_profile) // 2  # to do some rounding
            left_flux = np.nansum(spatial_profile[:mid])
            right_flux = np.nansum(spatial_profile[mid:])
            asym = (right_flux - left_flux) / (right_flux + left_flux + 1e-8) # so it never blows up, because it did.
    
            center.append(cen)
            width.append(wid)
            total_flux.append(denom)
            peak.append(pk)
            asymmetry.append(asym)
    
    A_spectra = []
    B_spectra = []
    AB_labels = []
    nods_num_labels = []
    
    # working in groups of 8 frames per classification to "follow" classic patterns but allow wiggle room
    for start in range(0, len(collapsed), 8): 
        end = min(start + 8, len(collapsed))
    
        group_spectra = collapsed[start:end]
        group_centers = np.array(center[start:end], dtype=float)
        group_peaks = np.array(peak[start:end], dtype=float)
        group_asym = np.array(asymmetry[start:end], dtype=float)
        group_widths = np.array(width[start:end], dtype=float)
        group_flux = np.array(total_flux[start:end], dtype=float)
        
        features = np.column_stack([group_centers, group_widths, group_peaks, group_asym, group_flux])

        # standardize features by scaling to unit variance, putting features on a more comparable scale
        scaler = StandardScaler()
        features = scaler.fit_transform(features)

        features[:, 0] *= 3.0  # weighting centroid feature more by multiplying each value by three, an overall 9x more influence
    
        # again, confirming we don't have bad frames (Kmeans won't take NaNs)
        if not np.all(np.isfinite(features)):
            print(f"Non-finite features in group {start}:{end}") 
            print(features)
            continue
    
        kmeans = KMeans(n_clusters=2, random_state=0, n_init=20).fit(features) # run clustering algorithm!
        labels = kmeans.labels_
    
        # to visualize values
        print(f"group {start}:{end}")
        print("raw centers: ", group_centers)
        print("raw peaks: ", group_peaks)
        print("raw asymmetry: ", group_asym)
        print("raw widths: ", group_widths)
        print("total flux: ", group_flux) 
        print("counts: ", [np.sum(labels == j) for j in range(2)])
    
        # breaking up clusters to groups A and B
        cluster_center_means = []
        for j in range(2):
            cluster_center_means.append(np.nanmean(group_centers[labels == j]))
        cluster_center_means = np.array(cluster_center_means)
    
        # assigning letter classification to numerical value
        label_A = np.argmin(cluster_center_means)   # A = 0
        label_B = np.argmax(cluster_center_means)   # B = 1
    
        print("cluster mean centers: ", cluster_center_means)
        print("label_A: ", label_A, "label_B: ", label_B)
    
        # more silly group organization
        for i, label in enumerate(labels):
            if label == label_A:
                A_spectra.append(group_spectra[i])
            else:
                B_spectra.append(group_spectra[i])
    
        # for plotting/pipeline purposes (these are inefficient but I didn't want to modify prior code)
        # can neglect everything below
        group_labels_AB = ['A' if l == label_A else 'B' for l in labels]
        AB_labels.extend(group_labels_AB)
        group_labels_num = [0 if l == label_A else 1 for l in labels]
        nods_num_labels.extend(group_labels_num)

    # mostly just outputs from here on out
    print("num A:", len(A_spectra))
    print("num B:", len(B_spectra))

    for j, (c, w, f, lab) in enumerate(zip(center, width, total_flux, AB_labels)):
        print(j, c, w, f, lab)
    
    A_frames, B_frames = [], []
    for frame, label in zip(frames, AB_labels):
        if label == 'A':
            A_frames.append(frame)
        else:
            B_frames.append(frame)
    
    return A_frames, B_frames, A_spectra, B_spectra, AB_labels, nods_num_labels, collapsed, center, peak, asymmetry, total_flux, width

## Run data for Testing Set Tau Boo!

In [ ]:
# Run once, this takes a bit to go over all frames

frames = []
filenames = []

# opening frames for Tau Boo target star
for i in range(129, 187):   # adjust for different data sets
    file = f"Test_Data/2017jun18/TauBoo/jun18s0{i}.fits"
    with fits.open(file) as hdul:
        filenames.append(file)
        frames.append(hdul[0].data.astype(float))

bad_frames = []
good_frames = []
good_filenames = []
good_idx = []

# Looking for corrupted frames (low exposure, image is basically flat)
for i, image in enumerate(frames):
    image = image[::-1]

    # to reduce hot pixel impact, finding within a percentile range
    p5 = np.nanpercentile(image, 5)
    p95 = np.nanpercentile(image, 95)
    robust_range = p95 - p5

    if robust_range < 4500:  # hardcoded visually for Tau Boo --- this will be more adaptive within pipeline
        bad_frames.append(filenames[i])
        print("Skipping low-exposure frame:", i, filenames[i])
        continue
    else:
        good_frames.append(image)
        good_filenames.append(filenames[i])
        good_idx.append(i)

print("Good frames:", len(good_frames), good_idx)
print("Bad frames:", len(bad_frames))

frames = good_frames
filenames = good_filenames

In [ ]:
# run classification!
A_frames, B_frames, A_spectra, B_spectra, AB_labels, nods_num_labels, collapsed, center, peak, asymmetry, total_flux, width = classify_AB_frames(frames)

In [ ]:
# organize frames and combine for visual purposes (separated due to longer runtime of prior cells)
A_frames = np.array(A_frames, dtype=float)
B_frames = np.array(B_frames, dtype=float)

print("A_frames shape:", A_frames.shape)
print("B_frames shape:", B_frames.shape)

sum_A = np.nanmedian(A_frames, axis=0)   # combined 2D A image
sum_B = np.nanmedian(B_frames, axis=0)   # combined 2D B image

In [ ]:
prof_A_1 = np.nanmedian(sum_A, axis=1)  # A spatial profile
prof_B_1 = np.nanmedian(sum_B, axis=1)  # B spatial profile

# plot profiles together for comparison!
plt.figure(figsize=(12, 4))
plt.plot(prof_A_1, label="Nod A", c="royalblue")
plt.plot(prof_B_1, label="Nod B", c="deeppink")
plt.legend()
plt.title("Tau Boo: Collapsed Spatial Profile with Trace Pairs")
plt.xlabel("Spatial Pixel Position")
plt.ylabel("Flux")
plt.grid(alpha=0.5)
plt.show()

## Metrics and Evaluation

In [ ]:
# visually classified nod labels for performance evaluation
A_orig = [0, 3, 7, 23, 24, 27, 31, 32, 35, 36, 52]
B_orig = [1, 2, 6, 17, 18, 21, 22, 25, 26, 30, 33, 34, 53]

# making sure to keep things aligned with dropped frames
# this is all just repetitive organization, had lots of issues with organization within the pipeline
manual_lookup = {}
for i in A_orig:
    manual_lookup[i] = 0

for i in B_orig:
    manual_lookup[i] = 1

manual_labels = []
kept_manual_idx = []

for orig_idx in good_idx:
    if orig_idx in manual_lookup:
        manual_labels.append(manual_lookup[orig_idx])
        kept_manual_idx.append(orig_idx)

manual_labels = np.array(manual_labels)

print("Filtered manual-label indices:", kept_manual_idx)
print("manual_labels:", manual_labels)
print("Number of labeled filtered frames:", len(manual_labels))

labeled_mask = np.array([orig_idx in manual_lookup for orig_idx in good_idx])

print("Labeled mask length:", len(labeled_mask))
print("Number of labeled filtered frames:", np.sum(labeled_mask))

In [ ]:
# function pulled from HW 3
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.

    Inputs
    cm: confusion matrix (2D numpy array)
    classes: list of class labels
    normalize: whether to normalize the matrix
    title: title of the plot
    cmap: color map for the plot
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    plt.figure(figsize=(7,6))
    print(cm)
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center", verticalalignment="center",
                 color="green" if i == j else "red", fontsize = 30)

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

In [ ]:
# metrics! with or without labels to compare, since we only have them if done manually
# also confusion matrix for comparison

def evaluate_clustering(X, true_labels=None, plot_cm=False):
    # runs KMeans clustering
    kmeans = KMeans(n_clusters=2, n_init=20, random_state=0)
    pred = kmeans.fit_predict(X)

    results = {}

    # Unsupervised clustering metric to see how well the clusters are separated
    if len(np.unique(pred)) > 1:  # if we have two clusters, do it
        results["silhouette"] = silhouette_score(X, pred)
    else:
        results["silhouette"] = np.nan

    # if we have labels for comparison (supervised method), calculate accuracy, precision, recall
    if true_labels is not None:
        # trying both label assignments
        acc1 = accuracy_score(true_labels, pred)
        acc2 = accuracy_score(true_labels, 1 - pred)

        if acc2 > acc1:
            pred = 1 - pred

        # compute all metrics!
        results["accuracy"] = accuracy_score(true_labels, pred)
        results["precision"] = precision_score(true_labels, pred)
        results["recall"] = recall_score(true_labels, pred)

        # confusion matrix
        cm = confusion_matrix(true_labels, pred)
        results["confusion_matrix"] = cm
        if plot_cm:
            plot_confusion_matrix(cm, ["Nod A", "Nod B"], normalize=True, title=f"Confusion Matrix: {feats}")

    return results

In [ ]:
# evaluate with different configurations
feature_dict = {"center": np.array(center), "width": np.array(width), "flux": np.array(total_flux), "peak": np.array(peak), "asym": np.array(asymmetry)}
feature_sets = [["center"], ["center", "width"], ["center", "flux"], ["center", "peak"], ["center", "asym"], ["center", "peak", "asym"]]

results = []
for feats in feature_sets:
    cols = []
    
    for feat in feats:
        cols.append(feature_dict[feat])
        
    X = np.column_stack(cols)
    
    # keep only rows with finite values and manual labels
    finite_mask = np.all(np.isfinite(X), axis=1)
    mask = finite_mask & labeled_mask

    X_use = X[mask]
    y_use = np.array([manual_lookup[idx] for idx in np.array(good_idx)[mask]])

    # calculate metrics
    metrics = evaluate_clustering(X_use, y_use, plot_cm=True)

    row = {"features": feats, **metrics}
    results.append(row)

for r in results:
    print(r)

# Trying on testing set - 51Peg

In [ ]:
# get frames

frames = []
filenames = []

for i in range(239, 282):
    file = f"Test_Data/2017sep07/51Peg/sep07s0{i}.fits"
    with fits.open(file) as hdul:
        filenames.append(file)
        frames.append(hdul[0].data.astype(float))

print("Number of raw frames:", len(frames))
print("Frame shape:", frames[0].shape)

In [ ]:
# repeating same steps from Tau Boo training set with testing set 51 Peg

bad_frames = []
good_frames = []
good_filenames = []
good_idx = []

for i, image in enumerate(frames):
    test_image = image[::-1]

    p5 = np.nanpercentile(test_image, 5)
    p95 = np.nanpercentile(test_image, 95)
    robust_range = p95 - p5
    print(robust_range)

    if robust_range < 3100: # hardcoded this again, can optimize later
        bad_frames.append(filenames[i])
        print("Skipping low-exposure frame:", i, filenames[i])
    else:
        good_frames.append(image)
        good_filenames.append(filenames[i])
        good_idx.append(i)

frames = good_frames
filenames = good_filenames

print("Good frames:", len(frames), good_idx)
print("Bad frames:", len(bad_frames))

In [ ]:
# run classification!
A_frames, B_frames, A_spectra, B_spectra, AB_labels, nods_num_labels, collapsed, center, peak, asymmetry, total_flux, width = classify_AB_frames(frames)

In [ ]:
# organize frames and combine for visual purposes (separated due to longer runtime of prior cells)
A_frames = np.array(A_frames, dtype=float)
B_frames = np.array(B_frames, dtype=float)

print("A_frames shape:", A_frames.shape)
print("B_frames shape:", B_frames.shape)

sum_A = np.nanmedian(A_frames, axis=0)   # combined 2D A image
sum_B = np.nanmedian(B_frames, axis=0)   # combined 2D B image

In [ ]:
prof_A_1 = np.nanmedian(sum_A, axis=1)  # A spatial profile
prof_B_1 = np.nanmedian(sum_B, axis=1)  # B spatial profile

# plot profiles together for comparison!
plt.figure(figsize=(12, 4))
plt.plot(prof_A_1, label="Nod A", c="royalblue")
plt.plot(prof_B_1, label="Nod B", c="deeppink")
plt.legend()
plt.title("51 Peg: Collapsed Spatial Profile with Trace Pairs")
plt.xlabel("Spatial Pixel Position")
plt.ylabel("Flux")
plt.grid(alpha=0.5)
plt.show()

In [ ]:
# evaluate with different configurations
feature_dict = {"center": np.array(center), "width": np.array(width), "flux": np.array(total_flux), "peak": np.array(peak), "asym": np.array(asymmetry)}
feature_sets = [["center"], ["center", "width"], ["center", "flux"], ["center", "peak"], ["center", "asym"], ["center", "peak", "asym"]]

results = []
for feats in feature_sets:
    cols = []
    
    for feat in feats:
        cols.append(feature_dict[feat])
        
    X = np.column_stack(cols)
    
    # keep only rows with finite values, not worrying about manual labels
    finite_mask = np.all(np.isfinite(X), axis=1)
    X_use = X[finite_mask]

    # calculate metrics
    metrics = evaluate_clustering(X_use, true_labels=None, plot_cm=False)

    row = {"features": feats, **metrics}
    results.append(row)

for r in results:
    print(r)